# Multilevel Monte Carlo for Option Pricing

Demonstrates MLMC for Asian option pricing using `FinancialOptionML` with geometric Brownian motion paths at multiple resolution levels.

In [1]:
using QuasiMC
import QuasiMC: Uniform

## Setup

GBM with 16 time steps at the finest level, coarsest level at 4 steps.

In [2]:
# GBM path model
dd = IIDStdUniform(16; seed=7)
tm = GeometricBrownianMotion(dd;
    volatility=0.2,
    start_price=100.0,
    interest_rate=0.05,
    t_final=1.0)

# Multilevel integrand
fml = FinancialOptionML(tm; d_coarsest=4)
display_table(
    [(level=l, time_steps=dimension_at_level(fml, l)) for l in 0:2];
    headers=["level", "time steps"])

level,time steps
0,4
1,8
2,16


## Exact Reference Value

In [3]:
# Geometric Asian call with Black-Scholes formula
fo_exact = FinancialOption(tm; option_type=:asian, strike_price=100.0,
                            asian_mean=:geometric)
exact = get_exact_value(fo_exact)
println("Exact geometric Asian option price: $(round(exact, digits=4))")

Exact geometric Asian option price: 5.8417


## Per-Level Variance Decay

In [4]:
using Statistics

let
    rows = NamedTuple{(:level, :E_PfPc, :Var_PfPc, :E_Pf)}[]
    for l in 0:2
        d_l = dimension_at_level(fml, l)
        x_level = randn(1000, d_l)
        Pc, Pf = ml_evaluate(fml, x_level, l)
        dP = Pf .- Pc
        push!(rows, (level=l, E_PfPc=mean(dP), Var_PfPc=var(dP), E_Pf=mean(Pf)))
    end
    display_table(rows;
        headers=["level", "E[Pf-Pc]", "Var[Pf-Pc]", "E[Pf]"],
        formatters=(
            E_PfPc=v -> string(round(v; digits=5)),
            Var_PfPc=v -> string(round(v; digits=5)),
            E_Pf=v -> string(round(v; digits=4))))
end

level,E[Pf-Pc],Var[Pf-Pc],E[Pf]
0,0.0,0.0,0.0
1,0.0,0.0,0.0
2,0.0,0.0,0.0
